In [1]:
# ------------------------------- #
# Imports
    # System
import os
import sys
import json
import shutil as sh
import glob
import re

    # System +
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
import pandas as pd

    # Constraints
from ase.constraints import FixAtoms

    # File io
from ase.io import read, write

    # Other
from ase.visualize.plot import plot_atoms
from ase.data.colors import jmol_colors
from ase.symbols import symbols2numbers

# Mine
from DFTUtils import write_vasp_settings, write_settings_json, get_strain, render_chg_slice
from DFTUtils import copy_files_from_DFTUtilities, remove_files, make_directories_from_list

notebook_path = os.path.dirname(__vsc_ipynb_file__)
plt.rcParams['figure.dpi'] = 200

# Bandstructure:

In [ ]:
# --------------------------------------------------------------- #
os.chdir(notebook_path)

# Read in initial structure:
struct = read('PRESS_0/relax.traj', format = 'traj')

# ----------------------------- #
# Change Default Vasp Settings
updated_vasp_settings = {'ediff': 1e-8,
                         'algo': 'Normal',
                         'nelm': 200,
                         'encut': 500,
                         'symprec': 1e-8,
                         'kpd': 40000,

                         # Calculation
                         'istart': 1,
                         'icharg': 1,
                         'laechg': True,

                         # Extras
                         'ivdw': 12,

                        # Magnetism
                        'ispin': 1}

# filter settings
band_settings = { # run_dft = True,
                  # directory_suffix = None,
                'path': 'GXMGZ',
                'npoints': 50}

In [22]:
import seekpath
from DFTUtils import interchange_atoms_ase_spglib

struct = read('/projects/p32212/NewUserIsMe/Displacive_Transitions_NEBs/NCN/HgNCN/Pressure_Relaxations/PRESS_0/relax.traj')
struct_spg = interchange_atoms_ase_spglib(struct)

path = seekpath.get_explicit_k_path(struct_spg)

def get_kpoints_on_path(struct, min_points):
    from DFTUtils import interchange_atoms_ase_spglib

    struct_spg = interchange_atoms_ase_spglib(struct)

    kpoint_distance = 0.025
    path = seekpath.get_explicit_k_path(struct_spg, reference_distance=kpoint_distance)
    while np.min(np.diff(np.array(path['explicit_segments']))) < min_points:
        kpoint_distance -= 0.01
        path = seekpath.get_explicit_k_path(struct_spg, reference_distance=kpoint_distance)

    return path['explicit_kpoints_abs']

kpoints = get_kpoints_on_path(struct, 50)

/home/ysx6266/.conda/envs/atomistic/lib/python3.12/site-packages/seekpath/hpkot/__init__.py:156: DeprecationWarning: dict interface is deprecated. Use attribute interface instead
  conv_lattice = dataset['std_lattice']
/home/ysx6266/.conda/envs/atomistic/lib/python3.12/site-packages/seekpath/hpkot/__init__.py:316: EdgeCaseWarning: aP lattice, but the k_gamma3 angle is almost equal to 90 degrees
  warnings.warn(


In [25]:
kpts = {'path': 'GXMGZ',
        'npoints': 50}

In [18]:
path = seekpath.get_explicit_k_path(struct_spg, reference_distance=0.01)
print(np.diff(np.array(path['explicit_segments'])))

[[61]
 [45]
 [50]
 [78]
 [67]
 [64]
 [76]]


/home/ysx6266/.conda/envs/atomistic/lib/python3.12/site-packages/seekpath/hpkot/__init__.py:156: DeprecationWarning: dict interface is deprecated. Use attribute interface instead
  conv_lattice = dataset['std_lattice']
/home/ysx6266/.conda/envs/atomistic/lib/python3.12/site-packages/seekpath/hpkot/__init__.py:316: EdgeCaseWarning: aP lattice, but the k_gamma3 angle is almost equal to 90 degrees
  warnings.warn(


In [ ]:
dirlist = [something]

make_directories_from_list(dirlist, delete=False)
for ind, directory in enumerate(dirlist): # GPa
    # Change directory
    os.chdir(directory)

    # Write Settings JSONs
    write_settings_json(band_settings, 'band_settings.json')

    remove_files(['myjob*', 'BFGS_hessian.json'])
    copy_files_from_DFTUtilities(['ASE_BandStructure.py', 'HPC_Submission_Scripts/JobSubmission_BandStructure.q'])

    # Submit Jobs
    !sbatch JobSubmission_BandStructure.q

    os.chdir('../')